# Treino do detector de placa (YOLO11n) no Kaggle

**Antes de rodar, confira (aba direita do notebook):**
1. **Data** -> `+ Add Input` -> os dois datasets (dados e código).
2. **Settings** -> **Accelerator** -> `GPU T4 x2` (ou `P100`).
3. **Settings** -> **Internet** -> `On`.

In [ ]:
!pip install -q ultralytics
import torch
print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## Localizar e linkar os dados

Não assume um caminho fixo — procura em `/kaggle/input/` por `annotations_v2.json` e pela pasta com mais `.jpg`, e cria `data/annotations_v2.json` e `data/images` (symlinks) em `/kaggle/working/TCC`, igual à estrutura local.

In [ ]:
import glob
from pathlib import Path

REPO_ROOT = Path("/kaggle/working/TCC")
(REPO_ROOT / "data" / "raw").mkdir(parents=True, exist_ok=True)

ann_matches = glob.glob("/kaggle/input/**/annotations_v2.json", recursive=True)
assert ann_matches, (
    "annotations_v2.json não encontrado em /kaggle/input — confira se o dataset "
    "com data/images + annotations_v2.json foi adicionado (aba Data, + Add Input)."
)
ann_path = Path(ann_matches[0]).resolve()
print("annotations_v2.json encontrado em:", ann_path)

jpg_dirs = {}
for p in glob.glob("/kaggle/input/**/*.jpg", recursive=True):
    d = Path(p).parent
    jpg_dirs[d] = jpg_dirs.get(d, 0) + 1
assert jpg_dirs, "nenhum arquivo .jpg encontrado em /kaggle/input"
images_dir = max(jpg_dirs, key=jpg_dirs.get)
print(f"pasta de imagens encontrada: {images_dir} ({jpg_dirs[images_dir]} arquivos .jpg)")

ann_link = REPO_ROOT / "data" / "raw" / "annotations_v2.json"
img_link = REPO_ROOT / "data" / "raw" / "images"
if not ann_link.exists():
    ann_link.symlink_to(ann_path)
if not img_link.exists():
    img_link.symlink_to(images_dir)

print("data/raw/annotations_v2.json ->", ann_link.resolve())
print("data/raw/images ->", img_link.resolve())

## Localizar e copiar o código (`src/` + `configs/`)

Mesma ideia: procura por `training/train.py` dentro de `/kaggle/input/` em vez de assumir um caminho fixo, e copia `src/` e `configs/` sem alterar nada dentro deles.

In [ ]:
import shutil

code_matches = glob.glob("/kaggle/input/**/training/train.py", recursive=True)
assert code_matches, (
    "train.py não encontrado em /kaggle/input — confira se o dataset com src/ + configs/ "
    "foi adicionado (aba Data, + Add Input)."
)
src_dir = Path(code_matches[0]).resolve().parents[1]  # .../src
project_dir = src_dir.parent
configs_dir = project_dir / "configs"
assert configs_dir.exists(), f"esperava configs/ ao lado de src/ em {project_dir}, não achei"

for name, srcp in (("src", src_dir), ("configs", configs_dir)):
    dst = REPO_ROOT / name
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(srcp, dst)

print("código copiado (sem alterações) para", REPO_ROOT)

## Checagem rápida antes de rodar tudo

In [ ]:
import json

with open(ann_link) as f:
    data = json.load(f)
print("imagens no annotations_v2.json:", len(data))
assert len(data) == 37099, f"esperava 37099 imagens, achei {len(data)} — dataset pode estar incompleto"

n_jpgs = len(list(img_link.glob("*.jpg")))
print("arquivos .jpg em data/raw/images:", n_jpgs)
assert n_jpgs >= 37000, f"esperava ~37101 imagens, achei {n_jpgs} — upload pode estar incompleto"

print("OK, dataset parece completo.")

## Preparar o dataset

`build_splits.py` já aplica as regras de filtragem por dentro (via `filters.py`: descarta imagem com `faulty=true` ou com qualquer placa `leg=0`/ilegível) e gera o split por câmera; `convert_annotations.py` gera `dataset_yolo/` com o `data.yaml` já apontando pro caminho certo do Kaggle (ele resolve o caminho em tempo de execução, não precisa editar nada).

In [ ]:
!python /kaggle/working/TCC/src/data_prep/build_splits.py

In [ ]:
!python /kaggle/working/TCC/src/data_prep/convert_annotations.py

## Teste rápido na GPU do Kaggle — NÃO PULE ISSO

Antes de comprometer horas de cota com o treino de verdade, roda esse teste (poucos minutos, 300 imagens, 5 épocas — usa a mesma `--smoke-test` de antes, só que agora na GPU real do Kaggle em vez do CPU local). Ele confirma duas coisas ao mesmo tempo:

1. **O pipeline inteiro roda de ponta a ponta no Kaggle** e gera um `.pt` de verdade.
2. **Os checkpoints realmente vão sendo gravados a cada época** — depois de rodar a célula abaixo, **enquanto ela ainda está executando**, abra o painel de arquivos da sessão (barra lateral do editor) e veja se `runs/detect/smoke_test/weights/last.pt` e `/kaggle/working/checkpoints/last.pt` aparecem e crescem/atualizam a cada época. Tente baixar um deles agora, só pra confirmar que o botão de download funciona numa sessão ainda ativa.

**Também anote quanto tempo cada época levou** (aparece no log, ex: `epoca 3 concluida (0.8 min desde o inicio)`) — dá pra estimar `minutos/época * 80` e comparar com as ~12h de sessão antes de rodar o treino real. Se `configs/train.yaml` (`epochs: 80`) parecer não caber com folga, ajuste esse número antes de seguir.

In [ ]:
!python /kaggle/working/TCC/src/training/train.py --smoke-test --n 300 --smoke-epochs 5 --live-copy-dir /kaggle/working/checkpoints

## Treino de verdade

`SMOKE_TEST = False` é o padrão em `train.py`, então isso já roda o dataset completo lendo os hiperparâmetros de `configs/train.yaml` (`imgsz=960, epochs=80, batch=16, patience=20, save_period=10`). O `--live-copy-dir` copia `last.pt`/`best.pt` pra `/kaggle/working/checkpoints` a cada época, além do local de sempre — **essa é a pasta que você deve olhar/baixar manualmente de tempos em tempos durante o treino**, sem esperar o commit terminar.

**Prefira Save Version -> Save & Run All (Commit)** em vez de rodar célula a célula com a aba aberta, pra não depender da conexão do navegador. Mesmo assim, volte periodicamente (mobile serve) e tente baixar `/kaggle/working/checkpoints/last.pt` pelo painel de arquivos da sessão — é a garantia real contra perder tudo nas 12h.

In [ ]:
!python /kaggle/working/TCC/src/training/train.py --live-copy-dir /kaggle/working/checkpoints

## Se a sessão cair antes de terminar

Baixe o que estiver em `/kaggle/working/checkpoints/last.pt` (ou dentro de `outputs/runs/detect/full_run/weights/`, se a aba Output tiver capturado). Para continuar de onde parou numa próxima sessão:

1. Suba a pasta do run inteira (`outputs/runs/detect/full_run/` — precisa do `last.pt` **e** do `args.yaml` que fica ao lado dele, não só o `.pt` sozinho) como um novo Kaggle Dataset (ou adicione ao dataset de código).
2. Repita as células de setup deste notebook (pip install, localizar dados, localizar código) normalmente.
3. No lugar da célula de treino, rode:

```python
!python /kaggle/working/TCC/src/training/train.py --resume /caminho/para/last.pt --live-copy-dir /kaggle/working/checkpoints
```

apontando `--resume` pro `last.pt` que você resgatou. O Ultralytics retoma a contagem de épocas de onde parou, usando a configuração salva junto (`args.yaml`).

## Deixar o `best.pt` fácil de baixar

Copia o resultado para a raiz de `/kaggle/working/`. Se o treino completou normalmente, usa o `best.pt` do run; se não achar (sessão cortada no meio), cai pro `last.pt` mais recente salvo em `--live-copy-dir` como alternativa.

In [ ]:
best = REPO_ROOT / "outputs" / "runs" / "detect" / "full_run" / "weights" / "best.pt"
live_copy_last = Path("/kaggle/working/checkpoints/last.pt")

if best.exists():
    shutil.copy2(best, "/kaggle/working/best.pt")
    print("Treino completo: /kaggle/working/best.pt (best.pt do run) — baixe pela aba Output/Data.")
elif live_copy_last.exists():
    shutil.copy2(live_copy_last, "/kaggle/working/best.pt")
    print(
        "[aviso] não achei o best.pt final (treino pode ter sido cortado) — usando o último "
        "checkpoint salvo em --live-copy-dir em vez disso."
    )
    print("/kaggle/working/best.pt (na verdade é o last.pt mais recente) — baixe pela aba Output/Data.")
else:
    raise RuntimeError(
        "não achei nem o best.pt do run nem nada em --live-copy-dir — confira os logs do treino acima."
    )